# S3 import of Survey Data and AQI Data

This is the import process for the data that is stored in S3. We ran a separate process to convert the data into a parquet file. We will be reading it from there. 

This data is the BRFSS (Behavioral Risk Factor Survey System) data that we will use to get information regarding people and their health responses. We will only grab a select number of the columns

In [4]:
!pip install s3fs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 198.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: botocore
    Found existing installation: botocore 1.35.99
    Uninstalling botocore-1.35.99:
      Successfully uninstalled botocore-1.35.99
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [s3fs]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
boto3 1.35.99 requires botocore<1.36.0,>=1.35.99, but you have botocore 1.38.27 which is incompatible.


In [7]:
import os
import pandas as pd
import s3fs  # required to read directly from S3

#Running individually for each year 
df = pd.read_parquet('s3://americasbreath123/LLCP2018.parquet', engine='pyarrow')
#df = pd.read_parquet('s3://americasbreath123/LLCP2019_subset.parquet', engine='pyarrow')
#df = pd.read_parquet('s3://americasbreath123/LLCP2020_subset.parquet', engine='pyarrow')
#df = pd.read_parquet('s3://americasbreath123/LLCP2021_subset.parquet', engine='pyarrow')
#df = pd.read_parquet('s3://americasbreath123/LLCP2022_subset.parquet', engine='pyarrow')
#df = pd.read_parquet('s3://americasbreath123/LLCP2023_subset.parquet', engine='pyarrow')

We have successfully read in the parquet file into deepnote!!!! This is a success. 

Now, we need to use the codebook code that was identified to get this dataset into a format that we can make observations on. 

In [307]:
# Re-import necessary libraries after environment reset
import re

# Reload the uploaded file
file_path = "../data/codebooks/FORMAT23.sas"
with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    print(f"Reading file : {file_path}")
    sas_text = f.read()
    #print(sas_text)

# Find all VALUE blocks
format_blocks = re.findall(r"(?i)value\s+(\w+)\s+(.*?);", sas_text, re.DOTALL)
#print(format_blocks[0])
#print( type(format_blocks) )

# Convert each block into a dictionary
format_dicts = {}
cleaned_lines = []
codebook_dict = {}
#format_blocks = format_blocks[0]
#item = format_blocks[0]

#print(item)
#print(type(item))
for item in format_blocks:
#for _ in range(0,1):     
    value = item[0]
    lines = item[1].strip().splitlines()
    #print(f"Value : {value}")
    #print(f"Lines : {lines}")
    dict = {}
    for line in lines:
        #print(f"Line : {line}")
        line_arr = line.split("=")
        dict[line_arr[0].strip().replace('"','')] = line_arr[1].strip().replace('"','')
    codebook_dict[value] = dict
    #break

print(codebook_dict)

Reading file : ../data/codebooks/FORMAT23.sas
{'ACE1HURT': {'.': 'Not asked or Missing', '.D': 'DK/NS', '.R': 'REFUSED', '1': 'Never', '2': 'Once', '3': 'More than once', '7': 'Dont know/Not Sure', '9': 'Refused'}, 'ACEADNED': {'.': 'Not asked or Missing', '.D': 'DK/NS', '.R': 'REFUSED', '1': 'Never', '2': 'A little of the time', '3': 'Some of the time', '4': 'Most of the time', '5': 'All of the time', '7': 'Dont know/Not sure', '9': 'Refused'}, 'ACEADSAF': {'.': 'Not asked or Missing', '.D': 'DK/NS', '.R': 'REFUSED', '1': 'Never', '2': 'A little of the time', '3': 'Some of the time', '4': 'Most of the time', '5': 'All of the time', '7': 'Dont know/Not sure', '9': 'Refused'}, 'ACEDEPRS': {'.': 'Not asked or Missing', '.D': 'DK/NS', '.R': 'REFUSED', '1': 'Yes', '2': 'No', '7': 'Dont know/Not Sure', '9': 'Refused'}, 'ACEDIVRC': {'.': 'Not asked or Missing', '.D': 'DK/NS', '.R': 'REFUSED', '1': 'Yes', '2': 'No', '7': 'Dont know/Not Sure', '8': 'Parents not married', '9': 'Refused'}, 'ACED

In [308]:
# Loop through the codebook and replace codes with descriptions
for col, mapping in codebook_dict.items():
    if col in df.columns:
        df[col] = df[col].astype(str).map(mapping).fillna(df[col])  # fallback to original if unmapped

# Show result
df.head()

,_STATE,_URBSTAT,SEXVAR,_SEX,_AGEG5YR,EDUCA,INCOME3,RENTHOM1,EMPLOY1,_STSTR,...,SMOKE100,SMOKDAY2,USENOW3,ECIGNOW2,EXERANY2,AVEDRNK3,PRIMINS1,MEDCOST1,CHECKUP1,PERSDOC3
0,Alabama,1.0,NaN,NaN,NaN,9.0,12.0,NaN,2.0,11011,...,3.0,4.0,NaN,NaN,2.0,NaN,99.0,2.0,1.0,1.0
1,Alabama,1.0,NaN,NaN,NaN,5.0,42.0,NaN,2.0,11011,...,3.0,1.0,NaN,NaN,2.0,NaN,3.0,2.0,8.0,2.0
2,Alabama,1.0,NaN,NaN,NaN,0.0,12.0,NaN,2.0,11011,...,3.0,1.0,NaN,NaN,1.0,NaN,1.0,2.0,1.0,1.0
3,Alabama,1.0,NaN,NaN,NaN,7.0,21.0,2.0,2.0,11011,...,3.0,1.0,0.0,1.0,1.0,2.0,99.0,2.0,1.0,1.0
4,Alabama,1.0,NaN,NaN,NaN,5.0,11.0,2.0,2.0,11011,...,3.0,1.0,NaN,NaN,1.0,NaN,7.0,2.0,1.0,2.0


In [309]:
df['Year'] = 2022
df = df.rename(columns={'_STATE': 'State'})

In [286]:
#Okay we have a clean dataframe with the codebook applied to it. 
#Now we need to bring in AQI data and attempt to join it to the dataframe.

#Load AQI data now
aqi = pd.read_csv('../data/aqi_data/annual_conc_by_monitor_2019.csv')
#aqi = pd.read_csv('../data/aqi_data/annual_conc_by_monitor_2020.csv')
#aqi = pd.read_csv('../data/aqi_data/annual_conc_by_monitor_2021.csv')
#aqi = pd.read_csv('../data/aqi_data/annual_conc_by_monitor_2022.csv')
#aqi = pd.read_csv('../data/aqi_data/annual_conc_by_monitor_2023.csv')

In [ ]:
#necessary columns are: State Code', 'County Code', 'Site Num','Parameter Name','Arithmetic Mean'

aqi_subset = aqi[['State Name', 'County Code', 'Site Num', 'Parameter Name','Sample Duration','Pollutant Standard', 'Arithmetic Mean','Arithmetic Standard Dev']]
aqi_subset = aqi_subset.rename(columns={
    'State Name': 'State',
    'County Code': 'County',
    'Site Num': 'Site',
    'Parameter Name': 'Parameter',
    'Arithmetic Mean': 'AQI'
}) 

# Filter to PM2.5 and Ozone only
aqi_subset = aqi_subset[aqi_subset['Parameter'].isin(['PM2.5 - Local Conditions', 'Ozone'])]

#Filter the Sample Duration to 24 HR BLK AVG and 8 HR RUN AVG BEGIN HOUR
aqi_subset = aqi_subset[aqi_subset['Sample Duration'].isin(['24-HR BLK AVG', '8-HR RUN AVG BEGIN HOUR'])]

In [292]:
# Group by state and pollutant
state_pollutants = aqi_subset.groupby(['State', 'Parameter', 'Sample Duration','Pollutant Standard'])['AQI'].mean().reset_index()


In [294]:
# Pivot to get one row per state with PM2.5 and Ozone
state_pollutants_pivot = state_pollutants.pivot(index='State', columns=['Parameter','Sample Duration','Pollutant Standard'], values='AQI').reset_index()
#state_pollutants.columns.name = None  # Remove the column group name

In [296]:
# Flatten multi-level columns
state_pollutants_pivot.columns = ['_'.join([str(i) for i in col if i]) for col in state_pollutants_pivot.columns.values]


In [298]:
state_pollutants_pivot = state_pollutants_pivot[['State','Ozone_8-HR RUN AVG BEGIN HOUR_Ozone 8-hour 2015','PM2.5 - Local Conditions_24-HR BLK AVG_PM25 24-hour 2024']]

In [300]:
#Now we will do the merge to the BRFSS dataframe 

merged_df = pd.merge(df, state_pollutants_pivot, on='State', how='left')

In [303]:
merged_df.to_csv('../data/merged_data/merged_brfss_aqi_2019.csv', index=False)
#merged_df.to_csv('../data/merged_data/merged_brfss_aqi_2020.csv', index=False)
#merged_df.to_csv('../data/merged_data/merged_brfss_aqi_2021.csv', index=False)
#merged_df.to_csv('../data/merged_data/merged_brfss_aqi_2022.csv', index=False)
#merged_df.to_csv('../data/merged_data/merged_brfss_aqi_2023.csv', index=False)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5bbfec53-798c-4f0c-81bd-e810c95d0334' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>